In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/playground-series-s5e11/sample_submission.csv
/kaggle/input/playground-series-s5e11/train.csv
/kaggle/input/playground-series-s5e11/test.csv
/kaggle/input/pss5e11-model0-oofs/oof_xgb_cv_0.9259837175238498.csv
/kaggle/input/pss5e11-model0-oofs/test_xgb_cv_0.9259837175238498.csv
/kaggle/input/loan-prediction-dataset-2025/loan_dataset_20000.csv


In [2]:
N_FOLDS = 5
SEED = 42

In [3]:
INPUT_DIR = '/kaggle/input/playground-series-s5e11'
train = pd.read_csv(f'{INPUT_DIR}/train.csv')
train_org = pd.read_csv('/kaggle/input/loan-prediction-dataset-2025/loan_dataset_20000.csv')
test = pd.read_csv(f'{INPUT_DIR}/test.csv')
TARGET = train.columns[-1]
test[TARGET] = -1
# combine

In [4]:
FEATURES = list(train.columns[1:-1])
print(f'FEATURES_{len(FEATURES)}: {FEATURES}, TARGET: {TARGET}')

FEATURES_11: ['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate', 'gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose', 'grade_subgrade'], TARGET: loan_paid_back


In [5]:
train_org = train_org[FEATURES + [TARGET]]
train_org[TARGET] = train_org[TARGET].astype('float64')

# train_org = train_org[FEATURES + [TARGET]].reset_index(drop=True)
# train     = train[FEATURES + [TARGET]].reset_index(drop=True)
# test      = test[FEATURES].reset_index(drop=True)         

# n_org = len(train_org)
# n_tr  = len(train)
# n_te  = len(test)

combine = pd.concat([train_org, train.drop(columns='id'), test.drop(columns='id')], axis=0)

In [6]:
CATS = []
NUMS = []

for c in FEATURES:
    t='CAT'
    if train[c].dtype=='object':
        CATS.append(c)
    else:
        NUMS.append(c)
        t='NUM'

    n = train[c].nunique()
    na = train[c].isna().sum()
    print(f'[{t}] {c} has {n} unique and {na} NA')

print('CATS:', CATS)
print('NUMS:', NUMS)

[NUM] annual_income has 119728 unique and 0 NA
[NUM] debt_to_income_ratio has 526 unique and 0 NA
[NUM] credit_score has 399 unique and 0 NA
[NUM] loan_amount has 111570 unique and 0 NA
[NUM] interest_rate has 1454 unique and 0 NA
[CAT] gender has 3 unique and 0 NA
[CAT] marital_status has 4 unique and 0 NA
[CAT] education_level has 5 unique and 0 NA
[CAT] employment_status has 5 unique and 0 NA
[CAT] loan_purpose has 8 unique and 0 NA
[CAT] grade_subgrade has 30 unique and 0 NA
CATS: ['gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose', 'grade_subgrade']
NUMS: ['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate']


In [7]:
CATS1 = []
SIZES = {}

for c in CATS:
    # if c in NUMS:
    #     n=f'{c}2'
    #     CATS1.append(n)
    combine[c],_ = combine[c].factorize()
    SIZES[c] = combine[c].max()+1

    # combine[c] = combine[c].astype('int32')
    # combine[n] = combine[n].astype('int32')
for c in NUMS:
    n=f'{c}2'
    combine[n] = combine[c].astype('category')
    CATS1.append(n)
    
print('NEW CATS:', CATS1)
print('CARDINALITY OF ALL CATS:', SIZES)

NEW CATS: ['annual_income2', 'debt_to_income_ratio2', 'credit_score2', 'loan_amount2', 'interest_rate2']
CARDINALITY OF ALL CATS: {'gender': 3, 'marital_status': 4, 'education_level': 5, 'employment_status': 5, 'loan_purpose': 8, 'grade_subgrade': 30}


In [8]:
from itertools import combinations

INTER = []

for col1, col2 in combinations(FEATURES, 2):
    new_col_name = f'{col1}_{col2}'
    INTER.append(new_col_name)
    for df in [combine]:
        df[new_col_name] = df[col1].astype(str) + '_' + df[col2].astype(str)
        
print(f'{len(INTER)} Features.')

55 Features.


In [10]:
def add_domain_features(df):
    """Add professional financial risk assessment features"""
    
    # Monthly income calculation
    df['monthly_income'] = df['annual_income'] / 12
    
    # Estimated monthly payment (using simple interest approximation)
    df['estimated_monthly_payment'] = (df['loan_amount'] * (df['interest_rate'] / 100)) / 12
    
    # Payment to income ratio (key lending metric)
    df['payment_to_income_ratio'] = df['estimated_monthly_payment'] / (df['monthly_income'] + 1)
    
    # Loan to annual income ratio
    df['loan_to_annual_income'] = df['loan_amount'] / (df['annual_income'] + 1)
    
    # High risk flag (industry standard thresholds)
    df['high_risk_flag'] = (
        (df['credit_score'] < 650) & 
        (df['debt_to_income_ratio'] > 0.43)
    ).astype(int)
    
    # Income adequacy
    df['income_adequacy'] = df['annual_income'] / (df['loan_amount'] + 1)
    
    # Estimated total existing debt
    df['estimated_total_debt'] = df['annual_income'] * df['debt_to_income_ratio']
    
    # Remaining income after payment
    df['remaining_income_after_payment'] = df['monthly_income'] - df['estimated_monthly_payment']
    
    return df


combine = add_domain_features(combine)


ROUND = []
rounding_levels = {'1s': 0, '10s': -1}

for col in ['annual_income', 'loan_amount']:
    for suffix, level in rounding_levels.items():
        new_col = f"{col}_ROUND_{suffix}"
        ROUND.append(new_col)
        for df in [combine]:
            df[new_col] = df[col].round(level).astype(int)

print(f"✓ Created {len(ROUND)} rounding features")

✓ Created 4 rounding features


In [11]:
train_org_n = combine.iloc[:len(train_org)]
train_n = combine.iloc[len(train_org):len(train)+len(train_org)]
test_n = combine.iloc[len(train)+len(train_org):]

In [12]:
TE = []
for c in FEATURES:
    tmp = train_org_n.groupby(c)[TARGET].mean()
    tmp_sum = train_org_n.groupby(c)[TARGET].sum()
    tmp_cnt = train_org_n.groupby(c).size()
    
    n = f'TE_{c}'
    n_s = f'TE_sum_{c}'
    n_c = f'TE_count_{c}'
    print(f'{n} , {n_c}', end='')
    tmp.name = n
    tmp_cnt.name = n_c
    tmp_sum.name = n_s

    stats = (pd.concat([tmp,  
                       # tmp_sum,
                        tmp_cnt
                      ]
                      , axis=1,).reset_index().rename(columns={'index': c})) 
    train_org_n = train_org_n.merge(stats, on=c, how='left')
    
    train_n = train_n.merge(stats, on=c, how='left')
    
    test_n = test_n.merge(stats, on=c, how='left')
    
    TE.append(n)
    TE.append(n_c)

TE_annual_income , TE_count_annual_incomeTE_debt_to_income_ratio , TE_count_debt_to_income_ratioTE_credit_score , TE_count_credit_scoreTE_loan_amount , TE_count_loan_amountTE_interest_rate , TE_count_interest_rateTE_gender , TE_count_genderTE_marital_status , TE_count_marital_statusTE_education_level , TE_count_education_levelTE_employment_status , TE_count_employment_statusTE_loan_purpose , TE_count_loan_purposeTE_grade_subgrade , TE_count_grade_subgrade

In [14]:
from sklearn.base import BaseEstimator, TransformerMixin

class TargetEncoder(BaseEstimator, TransformerMixin):
    """
    Target Encoder that supports multiple aggregation functions,
    internal cross-validation for leakage prevention, and smoothing.

    Parameters
    ----------
    cols_to_encode : list of str
        List of column names to be target encoded.

    aggs : list of str, default=['mean']
        List of aggregation functions to apply. Any function accepted by
        pandas' `.agg()` method is supported, such as:
        'mean', 'std', 'var', 'min', 'max', 'skew', 'nunique', 
        'count', 'sum', 'median'.
        Smoothing is applied only to the 'mean' aggregation.

    cv : int, default=5
        Number of folds for cross-validation in fit_transform.

    smooth : float or 'auto', default='auto'
        The smoothing parameter `m`. A larger value puts more weight on the 
        global mean. If 'auto', an empirical Bayes estimate is used.
        
    drop_original : bool, default=False
        If True, the original columns to be encoded are dropped.
    """
    def __init__(self, cols_to_encode, aggs=['mean'], cv=5, smooth='auto', drop_original=False):
        self.cols_to_encode = cols_to_encode
        self.aggs = aggs
        self.cv = cv
        self.smooth = smooth
        self.drop_original = drop_original
        self.mappings_ = {}
        self.global_stats_ = {}

    def fit(self, X, y):
        """
        Learn mappings from the entire dataset.
        These mappings are used for the transform method on validation/test data.
        """
        temp_df = X.copy()
        temp_df['target'] = y

        # Learn global statistics for each aggregation
        for agg_func in self.aggs:
            self.global_stats_[agg_func] = y.agg(agg_func)

        # Learn category-specific mappings
        for col in self.cols_to_encode:
            self.mappings_[col] = {}
            for agg_func in self.aggs:
                mapping = temp_df.groupby(col)['target'].agg(agg_func)
                self.mappings_[col][agg_func] = mapping
        
        return self

    def transform(self, X):
        """
        Apply learned mappings to the data.
        Unseen categories are filled with global statistics.
        """
        X_transformed = X.copy()
        for col in self.cols_to_encode:
            for agg_func in self.aggs:
                new_col_name = f'TE_{col}_{agg_func}'
                map_series = self.mappings_[col][agg_func]
                X_transformed[new_col_name] = X[col].map(map_series)
                X_transformed[new_col_name].fillna(self.global_stats_[agg_func], inplace=True)
        
        if self.drop_original:
            X_transformed.drop(columns=self.cols_to_encode, inplace=True)
            
        return X_transformed

    def fit_transform(self, X, y):
        """
        Fit and transform the data using internal cross-validation to prevent leakage.
        """
        # First, fit on the entire dataset to get global mappings for transform method
        self.fit(X, y)

        # Initialize an empty DataFrame to store encoded features
        encoded_features = pd.DataFrame(index=X.index)
        
        kf = StratifiedKFold(n_splits=self.cv, shuffle=True, random_state=42)

        for train_idx, val_idx in kf.split(X, y):
            X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
            X_val = X.iloc[val_idx]
            
            temp_df_train = X_train.copy()
            temp_df_train['target'] = y_train

            for col in self.cols_to_encode:
                # --- Calculate mappings only on the training part of the fold ---
                for agg_func in self.aggs:
                    new_col_name = f'TE_{col}_{agg_func}'
                    
                    # Calculate global stat for this fold
                    fold_global_stat = y_train.agg(agg_func)
                    
                    # Calculate category stats for this fold
                    mapping = temp_df_train.groupby(col)['target'].agg(agg_func)

                    # --- Apply smoothing only for 'mean' aggregation ---
                    if agg_func == 'mean':
                        counts = temp_df_train.groupby(col)['target'].count()
                        
                        m = self.smooth
                        if self.smooth == 'auto':
                            # Empirical Bayes smoothing
                            variance_between = mapping.var()
                            avg_variance_within = temp_df_train.groupby(col)['target'].var().mean()
                            if variance_between > 0:
                                m = avg_variance_within / variance_between
                            else:
                                m = 0  # No smoothing if no variance between groups
                        
                        # Apply smoothing formula
                        smoothed_mapping = (counts * mapping + m * fold_global_stat) / (counts + m)
                        encoded_values = X_val[col].map(smoothed_mapping)
                    else:
                        encoded_values = X_val[col].map(mapping)
                    
                    # Store encoded values for the validation fold
                    encoded_features.loc[X_val.index, new_col_name] = encoded_values.fillna(fold_global_stat)

        # Merge with original DataFrame
        X_transformed = X.copy()
        for col in encoded_features.columns:
            X_transformed[col] = encoded_features[col]
            
        if self.drop_original:
            X_transformed.drop(columns=self.cols_to_encode, inplace=True)
            
        return X_transformed

In [16]:
X = train_n.drop(columns=[TARGET])
y = train_n[TARGET]
print(X.shape)

(593994, 105)


In [17]:
oofs0 = pd.read_csv('/kaggle/input/pss5e11-model0-oofs/oof_xgb_cv_0.9259837175238498.csv')
test0 = pd.read_csv('/kaggle/input/pss5e11-model0-oofs/test_xgb_cv_0.9259837175238498.csv')

In [18]:
from scipy.special import logit, expit

oofs0_logits = logit(oofs0['loan_paid_back'])
test0_logits = logit(test0['loan_paid_back'])

POSITIVE_LOGIT =  2.0
NEGATIVE_LOGIT = -2.0

true_logits = np.where(y == 1, POSITIVE_LOGIT, NEGATIVE_LOGIT)
residuals = true_logits - oofs0_logits

In [19]:
from xgboost import XGBClassifier, XGBRegressor
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'max_depth': 5,
    'colsample_bytree': 0.5,
    'subsample': 0.8,
    'n_estimators': 10000,
    'learning_rate': 0.01,
    'early_stopping_rounds': 100,
    'random_state': 42,
    'n_jobs': -1,
    'device': 'cuda',
    'enable_categorical': True,
}
params_residuals = params.copy()
params_residuals.update({
    'objective':'reg:squarederror',
    'eval_metric': 'rmse',
})
params2 = {     'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'learning_rate': 0.01,
    'max_depth': 6,
    'min_child_weight': 3,
    'colsample_bytree': 0.3,
    'subsample': 0.6,
    'reg_alpha': 0.5,
    'reg_lambda': 2.0,
    'n_estimators': 10000,
    'early_stopping_rounds': 200,
    'random_state': SEED,
    'n_jobs': -1,
    'enable_categorical': True,
    'device': 'cuda',
                      }
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
print(f'X shape: {X.shape}')

# X = X.iloc[:1000]
# y = y.iloc[:1000]

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(test))


for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print(f'--- Fold {fold}/{N_FOLDS} ---')
    
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    residuals_train = residuals.iloc[train_idx]
    residuals_val = residuals.iloc[val_idx]
    # X_test = test[FEATURES].copy()
    X_test = test_n.drop(columns=TARGET).copy()

    TE = TargetEncoder(cols_to_encode=INTER+CATS1, cv=5, smooth='auto', aggs=['mean', 'count'], drop_original=True)
    X_train = TE.fit_transform(X_train, y_train)
    X_val = TE.transform(X_val)
    X_test = TE.transform(X_test)

    TE2 = TargetEncoder(cols_to_encode=ROUND, cv=5, smooth='auto', aggs=['mean', 'count'], drop_original=False)
    X_train = TE2.fit_transform(X_train, y_train)
    X_val = TE2.transform(X_val)
    X_test = TE2.transform(X_test)
    
    # for c in CATS+CATS1:
    #     X_train[c] = X_train[c].astype('category')
    #     X_val[c] = X_val[c].astype('category')
    #     X_test[c] = X_test[c].astype('category')
    print('training shape :', X_train.shape)
    print(f'Residual stats - Mean: {residuals_train.mean():.3f}, Std: {residuals_train.std():.3f}')

    model = XGBClassifier(**params)
    
    model.fit(X_train, y_train,
              eval_set=[(X_val, y_val)],
              verbose=1000)

    val_preds = model.predict_proba(X_val)[:,1]
    oof_preds[val_idx] = val_preds
    
    fold_score = roc_auc_score(y_val, val_preds)
    print(f'Fold {fold} AUC: {fold_score:.4f}')
    test_preds += model.predict_proba(X_test)[:,1] / N_FOLDS

# test_preds = expit(test_preds+test0_logits)
overall_auc = roc_auc_score(y, oof_preds)
print(f'====================')
print(f'Overall OOF AUC: {overall_auc:.4f}')
print(f'====================')

X shape: (593994, 105)
--- Fold 1/5 ---
training shape : (475195, 173)
Residual stats - Mean: -0.993, Std: 1.929
[0]	validation_0-auc:0.90885
[1000]	validation_0-auc:0.92697
[1595]	validation_0-auc:0.92721
Fold 1 AUC: 0.9272
--- Fold 2/5 ---
training shape : (475195, 173)
Residual stats - Mean: -1.001, Std: 1.932
[0]	validation_0-auc:0.91527
[1000]	validation_0-auc:0.92791
[1342]	validation_0-auc:0.92798
Fold 2 AUC: 0.9280
--- Fold 3/5 ---
training shape : (475195, 173)
Residual stats - Mean: -0.991, Std: 1.923
[0]	validation_0-auc:0.90871
[1000]	validation_0-auc:0.92574
[1254]	validation_0-auc:0.92579
Fold 3 AUC: 0.9258
--- Fold 4/5 ---
training shape : (475195, 173)
Residual stats - Mean: -0.998, Std: 1.937
[0]	validation_0-auc:0.90811
[1000]	validation_0-auc:0.92684
[1203]	validation_0-auc:0.92687
Fold 4 AUC: 0.9269
--- Fold 5/5 ---
training shape : (475196, 173)
Residual stats - Mean: -1.001, Std: 1.923
[0]	validation_0-auc:0.91473
[1000]	validation_0-auc:0.92636
[1588]	validation_

In [21]:
pd.DataFrame({'id': train.id, TARGET: oof_preds}).to_csv(f'oof_xgb_cv_{overall_auc}.csv', index=False)
pd.DataFrame({'id': test.id, TARGET: test_preds}).to_csv(f'test_xgb_cv_{overall_auc}.csv', index=False)